# Demo 04f : Context Engineering

What actually occupies the context window, and the four strategies for
controlling it : write, select, compress and isolate.

Every section measures. The numbers are what make the point, so run the cells
rather than reading them.

Runs against a local model by default.

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [1]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

model=qwen3.5:2b


## 1. Where the tokens go

The window holds four things : the system prompt, the tool definitions, the
history and the question. Asking the model the same question four times, adding
one component each time, prices each of them by difference.

In [2]:
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool


@tool
def lookup_ticket(ticket_id: str) -> str:
    """Return the status, owner and last update of a service desk ticket."""
    return "open, owned by second line, last updated yesterday"


@tool
def search_kb(query: str) -> str:
    """Search the knowledge base and return the most relevant article."""
    return "KB-114 : resetting a locked account"


SYSTEM = ("You are the service desk assistant for a mid-sized bank. "
          "Answer from the knowledge base where you can. Escalate anything "
          "touching payments to second line. Never guess a ticket number.")

history = []
for i in range(6):
    history += [HumanMessage(f"Question {i} about a locked account."),
                AIMessage(f"Answer {i}, referring to the standard reset procedure.")]

question = [HumanMessage("Is ticket 4471 still open?")]


def cost(messages, tools=None):
    model = llm.bind_tools(tools) if tools else llm
    with get_usage_metadata_callback() as cb:
        model.invoke(messages)
    return list(cb.usage_metadata.values())[0]["input_tokens"]


bare = cost(question)
system = cost([SystemMessage(SYSTEM)] + question)
tools = cost([SystemMessage(SYSTEM)] + question, [lookup_ticket, search_kb])
full = cost([SystemMessage(SYSTEM)] + history + question, [lookup_ticket, search_kb])

print(f"question only          {bare:5d}")
print(f"+ system prompt        {system:5d}   (+{system - bare})")
print(f"+ two tool definitions {tools:5d}   (+{tools - system})")
print(f"+ six turns of history {full:5d}   (+{full - tools})")

question only             22
+ system prompt           64   (+42)
+ two tool definitions   386   (+322)
+ six turns of history   560   (+174)


The question is a handful of tokens. Everything else is overhead that the loop
resends on every single pass, and only the last line grows as the agent works.

Two tools is a small agent. A dozen MCP servers puts hundreds of definitions in
front of the model before it has read the question.

## 2. Compress : full, window and summary

Three ways to carry a long conversation forward. The question at the end asks
about something said early on, so the cost is only half the result : whether the
strategy still answers correctly is the other half.

In [3]:
long_history = []
long_history += [HumanMessage("My account number is 88-4413-02. Remember it."),
                 AIMessage("Noted, 88-4413-02.")]
for i in range(14):
    long_history += [HumanMessage(f"Unrelated follow-up number {i} about opening hours."),
                     AIMessage(f"The branch is open nine to five. Note {i}.")]

recall = [HumanMessage("What is my account number?")]


def ask(messages):
    with get_usage_metadata_callback() as cb:
        reply = llm.invoke(messages)
    used = list(cb.usage_metadata.values())[0]["input_tokens"]
    return used, reply.content.strip().replace("\n", " ")[:70]


# full : every message, every time
t_full, a_full = ask([SystemMessage(SYSTEM)] + long_history + recall)

# window : the last six messages only
t_win, a_win = ask([SystemMessage(SYSTEM)] + long_history[-6:] + recall)

# summary : compress everything but the last four
older = "\n".join(f"{m.type}: {m.content}" for m in long_history[:-4])
summary = llm.invoke([HumanMessage(
    "Summarise this conversation in three sentences, keeping any numbers "
    "the customer gave:\n\n" + older)]).content
t_sum, a_sum = ask([SystemMessage(SYSTEM),
                    SystemMessage("Earlier conversation: " + summary)]
                   + long_history[-4:] + recall)

for name, tokens, answer in [("full", t_full, a_full),
                             ("window", t_win, a_win),
                             ("summary", t_sum, a_sum)]:
    print(f"{name:8s} {tokens:5d} tokens   {answer}")

full       574 tokens   Your account number is **88-4413-02**.
window     165 tokens   I do not have access to your personal account information or the abili
summary    220 tokens   Your account number is **88-4413-02**.


Window memory is the cheapest and has forgotten the account number. Summary
memory costs an extra model call to produce the summary, then stays cheap on
every turn after it, and usually still has the number because the instruction
asked for it.

That instruction is the whole design decision. A summary prompt that does not
say what to preserve will drop exactly the detail the next question needs.

## 3. Select : retrieval instead of the whole corpus

Retrieval reaches documents that do not fit. It is also a way to
keep the ones that would fit out of the window.

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

kb = [
    "KB-101 : card blocked after three wrong PIN entries, unblock at any branch.",
    "KB-114 : resetting a locked online banking account requires an ID check.",
    "KB-120 : mortgage interest is fixed for the agreed term and cannot change.",
    "KB-131 : a SEPA transfer arrives the next working day, cut-off is 16:00.",
    "KB-142 : lost cards are cancelled immediately on the phone line.",
    "KB-155 : joint account holders both have to sign a closure request.",
    "KB-160 : standing orders can be changed in the app up to a day ahead.",
    "KB-177 : a chargeback has to be raised within eight weeks of the debit.",
]
store = InMemoryVectorStore.from_documents(
    [Document(page_content=t) for t in kb], make_embeddings())

ask_kb = "How long do I have to raise a chargeback?"

# everything : paste the whole knowledge base into the prompt
t_all, a_all = ask([SystemMessage("Answer only from the context.\n\n" + "\n".join(kb)),
                    HumanMessage(ask_kb)])

# selected : the two nearest articles
hits = store.similarity_search(ask_kb, k=2)
t_sel, a_sel = ask([SystemMessage("Answer only from the context.\n\n"
                                  + "\n".join(d.page_content for d in hits)),
                    HumanMessage(ask_kb)])

print(f"whole corpus   {t_all:5d} tokens   {a_all}")
print(f"top two        {t_sel:5d} tokens   {a_sel}")
print()
print("selected:", *[d.page_content[:40] for d in hits], sep="\n  ")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

whole corpus     196 tokens   You have to raise a chargeback within eight weeks of the debit.
top two           76 tokens   You have eight weeks to raise a chargeback after the date of the debit

selected:
  KB-177 : a chargeback has to be raised w
  KB-160 : standing orders can be changed 


Eight short articles fit comfortably, and both answers are right, so the only
difference here is cost. Multiply the corpus by a thousand and the first option
stops being possible at all, while the second does not change.

The failure mode to watch is the one this cell is too small to show : if the
retriever returns two articles that contradict each other, the model picks one
and says it confidently.

## 4. Isolate : a sub-agent keeps its mess to itself

A sub-agent is usually introduced as division of labour. It is also a context
strategy, and that is often the better reason to reach for one.

In [5]:
from langchain.agents import create_agent

triage = create_agent(llm, [lookup_ticket, search_kb])

task = "Check ticket 4471 and find the article that covers a locked account."

# inline : the whole exchange, tool calls included, stays in the caller's list
inline = triage.invoke({"messages": [SystemMessage(SYSTEM), HumanMessage(task)]})
inline_messages = inline["messages"]

# isolated : the sub-agent runs, and only its conclusion comes back
isolated = [SystemMessage(SYSTEM), HumanMessage(task),
            AIMessage(inline_messages[-1].content)]

print(f"inline   {len(inline_messages):2d} messages")
for m in inline_messages:
    print(f"           {m.type:9s} {str(m.content)[:52]}")
print(f"isolated {len(isolated):2d} messages")

print()
print(f"inline   {cost(inline_messages):5d} tokens carried forward")
print(f"isolated {cost(isolated):5d} tokens carried forward")

inline    7 messages
           system    You are the service desk assistant for a mid-sized b
           human     Check ticket 4471 and find the article that covers a
           ai        
           tool      open, owned by second line, last updated yesterday
           ai        
           tool      KB-114 : resetting a locked account
           ai        I found the article covering a locked account in you
isolated  3 messages



inline     219 tokens carried forward
isolated   116 tokens carried forward


The tool calls, the tool results and any wrong turn the sub-agent took are all
in the inline list, and all of them are resent on every later turn. The isolated
version carries the answer and nothing else.

The cost is that the caller cannot see how the answer was reached, which is why
the trace matters more once work is split this way. That is module 09.

## 5. The tool result that eats the window

The most common cause of context exhaustion in a working agent, and the easiest
to fix.

In [6]:
import json as _json

RECORD = {"ticket": "4471", "status": "open", "owner": "second line",
          "opened": "2026-08-31T09:14:00Z", "priority": "P3",
          "customer": {"id": "88-4413-02", "segment": "retail", "since": 2011},
          "history": [{"at": f"2026-09-0{d}T10:0{d}:00Z", "by": "agent",
                       "note": "Contacted the customer, no answer, will retry."}
                      for d in range(1, 5)],
          "attachments": ["screenshot.png", "log.txt"], "sla_breached": False}


@tool
def get_ticket_raw(ticket_id: str) -> str:
    """Return the full ticket record."""
    return _json.dumps(RECORD, indent=2)


@tool
def get_ticket_summary(ticket_id: str) -> str:
    """Return the status, owner and whether the SLA is breached."""
    return (f"{RECORD['ticket']}: {RECORD['status']}, owned by "
            f"{RECORD['owner']}, SLA breached: {RECORD['sla_breached']}")


raw = get_ticket_raw.invoke({"ticket_id": "4471"})
summary = get_ticket_summary.invoke({"ticket_id": "4471"})

after_raw = [SystemMessage(SYSTEM), HumanMessage("Is 4471 open?"),
             AIMessage(""), HumanMessage("Tool result: " + raw),
             HumanMessage("And is it breaching the SLA?")]
after_sum = [SystemMessage(SYSTEM), HumanMessage("Is 4471 open?"),
             AIMessage(""), HumanMessage("Tool result: " + summary),
             HumanMessage("And is it breaching the SLA?")]

print(f"raw record     {len(raw):5d} characters   {cost(after_raw):5d} tokens on the next turn")
print(f"summary        {len(summary):5d} characters   {cost(after_sum):5d} tokens on the next turn")

raw record       848 characters     473 tokens on the next turn


summary           53 characters     107 tokens on the next turn


Both tools answer the question. One of them costs its tokens again on every turn
that follows, for the whole rest of the run.

Three fixes, in order of preference : return the answer rather than the payload,
paginate and let the model ask for more, or write the payload somewhere and
return a handle to it. The rule is that a tool returns what the model needs in
order to decide, and no more.

## What to take away

- The window is a fixed budget and the loop spends it
- Write : the history lives outside the window, behind a checkpointer
- Select : retrieval puts only the relevant passage in the window
- Compress : a summary keeps the substance, if the prompt says what to keep
- Isolate : a sub-agent's failed attempts never reach the caller
- A tool result is charged again on every turn that follows it

`exercise27` is the same problem on an agent that has not had any of this done
to it.